# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [ ]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [ ]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

DB = "prices.db"

In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [ ]:
get_ticket_price("Paris")

In [ ]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools

In [ ]:

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## A bit more about what Gradio actually does:

1. Gradio constructs a frontend Svelte app based on our Python description of the UI
2. Gradio starts a server built upon the Starlette web framework listening on a free port that serves this Svelte app
3. Gradio creates backend routes for our callbacks, like chat(), which calls our functions

And of course when Gradio generates the frontend app, it ensures that the the Submit button calls the right backend route.

That's it!

It's simple, and it has a result that feels magical.

# Let's go multi-modal!!

We can use gpt-image-1-mini, the image generation model behind GPT-5, to make us some images

Let's put this in a function called artist.

### Price alert: each time I generate an image it costs about 3 cents - don't go crazy with images!

In [ ]:
# Some imports for handling images

import base64
from io import BytesIO
from PIL import Image

In [ ]:
def artist(city):
    image_response = openai.images.generate(
            model="gpt-image-1-mini",
            prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
            size="1024x1024",
            n=1,
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [ ]:
image = artist("New York City")
display(image)

In [ ]:
def talker(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",    # Also, try replacing onyx with alloy or coral
      input=message
    )
    return response.content

## Let's bring this home:

1. A multi-modal AI assistant with image and audio generation
2. Tool callling with database lookup
3. A step towards an Agentic workflow


In [ ]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    if cities:
        image = artist(cities[0])
    
    return history, voice, image


In [ ]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

## The 3 types of Gradio UI

`gr.Interface` is for standard, simple UIs

`gr.ChatInterface` is for standard ChatBot UIs

`gr.Blocks` is for custom UIs where you control the components and the callbacks

In [ ]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True, auth=("ed", "bananas"))

# Exercises and Business Applications

Add in more tools - perhaps to simulate actually booking a flight. A student has done this and provided their example in the community contributions folder.

Next: take this and apply it to your business. Make a multi-modal AI assistant with tools that could carry out an activity for your work. A customer support assistant? New employee onboarding assistant? So many possibilities! Also, see the week2 end of week Exercise in the separate Notebook.

## Exercise solution: an assistant that can really book a flight

Four tools now: quote a price, book a seat, look a booking up, and cancel it. Bookings go into the
same SQLite database as the prices, so a reference that comes back really exists.

Two things keep this tidy as the number of tools grows: a small `schema()` helper builds the JSON
descriptions, and `handle_tool_calls` dispatches through a dictionary rather than a chain of ifs.

In [ ]:
# EXERCISE SOLUTION: more tools, so the assistant can actually book and cancel a flight
#
# The database gains a bookings table, and the assistant gains four tools. Everything is
# dispatched through one dictionary, so adding the next tool is two lines of work.

import random
import shutil
import string
import subprocess

DB = "prices.db"

# Pictures need OpenAI; without a key we still chat with a local model and talk with 'say'
MULTIMODAL = bool(os.getenv("OPENAI_API_KEY"))
if not MULTIMODAL:
    print("No OPENAI_API_KEY: chatting with local llama3.2, speaking with this Mac, and no pictures")
    openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    MODEL = "llama3.2"

with sqlite3.connect(DB) as conn:
    conn.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.execute("""CREATE TABLE IF NOT EXISTS bookings (
                        reference TEXT PRIMARY KEY,
                        passenger_name TEXT,
                        city TEXT,
                        travel_date TEXT,
                        price REAL)""")


def say_locally(text):
    """No OpenAI key? On a Mac the built-in 'say' command can read the reply out instead."""
    if not shutil.which("say"):
        return None
    path = "/tmp/flightai_reply.wav"
    subprocess.run(["say", "--data-format=LEI16@22050", "-o", path, text[:700]], check=True, timeout=120)
    return path


def price_of(city):
    """The raw price for a city, or None if we don't fly there."""
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
    return result[0] if result else None


def get_ticket_price(destination_city):
    print(f"DATABASE TOOL CALLED: getting the price for {destination_city}", flush=True)
    price = price_of(destination_city)
    return f"Ticket price to {destination_city} is ${price}" if price else "No price data available for this city"


def book_flight(destination_city, passenger_name, travel_date):
    """Reserve a seat and return the booking reference."""
    print(f"DATABASE TOOL CALLED: booking {passenger_name} to {destination_city} on {travel_date}", flush=True)
    price = price_of(destination_city)
    if price is None:
        return f"FlightAI does not fly to {destination_city}, so no booking was made."
    reference = "FA-" + "".join(random.choices(string.ascii_uppercase + string.digits, k=6))
    with sqlite3.connect(DB) as conn:
        conn.execute(
            'INSERT INTO bookings (reference, passenger_name, city, travel_date, price) VALUES (?, ?, ?, ?, ?)',
            (reference, passenger_name, destination_city.lower(), travel_date, price),
        )
        conn.commit()
    return (f"Booked {passenger_name} to {destination_city} on {travel_date} for ${price}. "
            f"The booking reference is {reference}.")


def get_booking(reference):
    """Look up one booking by its reference."""
    print(f"DATABASE TOOL CALLED: looking up booking {reference}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            'SELECT passenger_name, city, travel_date, price FROM bookings WHERE reference = ?',
            (reference.upper(),),
        )
        result = cursor.fetchone()
    if not result:
        return f"There is no booking with reference {reference}."
    name, city, date, price = result
    return f"{reference.upper()}: {name} to {city.title()} on {date}, ${price}"


def cancel_booking(reference):
    """Cancel a booking and say whether there was anything to cancel."""
    print(f"DATABASE TOOL CALLED: cancelling booking {reference}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.execute('DELETE FROM bookings WHERE reference = ?', (reference.upper(),))
        conn.commit()
        cancelled = cursor.rowcount
    return f"Booking {reference.upper()} is cancelled." if cancelled else f"There is no booking with reference {reference}."


def schema(name, description, properties):
    """Build the JSON schema for a tool from its properties; every property is required."""
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": list(properties),
                "additionalProperties": False,
            },
        },
    }


city_property = {"destination_city": {"type": "string", "description": "The city the customer wants to travel to"}}
reference_property = {"reference": {"type": "string", "description": "The booking reference, for example FA-7K2QX1"}}

tools = [
    schema("get_ticket_price", "Get the price of a return ticket to the destination city.", city_property),
    schema(
        "book_flight",
        "Book a return flight for a passenger. Only call this once you know the destination, the passenger's name and the travel date.",
        {
            **city_property,
            "passenger_name": {"type": "string", "description": "The full name of the passenger"},
            "travel_date": {"type": "string", "description": "The date of travel, as YYYY-MM-DD"},
        },
    ),
    schema("get_booking", "Look up an existing booking by its reference.", reference_property),
    schema("cancel_booking", "Cancel an existing booking by its reference.", reference_property),
]

tool_functions = {
    "get_ticket_price": get_ticket_price,
    "book_flight": book_flight,
    "get_booking": get_booking,
    "cancel_booking": cancel_booking,
}

system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
You can quote prices, book flights, look up bookings and cancel them.
Before booking, make sure you know the destination city, the passenger's name and the travel date;
ask for whatever is missing rather than inventing it. Always quote the booking reference back to the customer.
Always be accurate. If you don't know the answer, say so.
"""


def handle_tool_calls(message):
    """Run every tool the model asked for, and remember which cities came up."""
    responses, cities = [], []
    for tool_call in message.tool_calls:
        name = tool_call.function.name
        function = tool_functions.get(name)
        if function is None:
            content = f"FlightAI has no tool called {name}."
        else:
            try:
                arguments = json.loads(tool_call.function.arguments)
                content = function(**arguments)
                if "destination_city" in arguments:
                    cities.append(arguments["destination_city"])
            except Exception as error:
                content = f"The {name} tool failed: {type(error).__name__}: {error}"
        responses.append({"role": "tool", "content": content, "tool_call_id": tool_call.id})
    return responses, cities


def chat(history):
    """Reply to the conversation so far, and return the reply, a voice and a picture."""
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses, called_with = handle_tool_calls(message)
        cities += called_with
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role": "assistant", "content": reply}]

    # Image generation needs OpenAI, but this machine can still do the talking
    voice = talker(reply) if MULTIMODAL else say_locally(reply)
    image = artist(cities[0]) if cities and MULTIMODAL else None
    return history, voice, image

In [ ]:
# The same Blocks UI as above, now backed by an assistant that can really book a flight

def put_message_in_chatbot(message, history):
    return "", history + [{"role": "user", "content": message}]


with gr.Blocks(title="FlightAI") as ui:
    with gr.Row():
        chatbot = gr.Chatbot(
            height=500,
            type="messages",
            value=[{"role": "assistant", "content": "Welcome to FlightAI. Where would you like to fly?"}],
        )
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:", submit_btn=True)

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True)

Try: *"What does a flight to Tokyo cost?"*, then *"Book it for Alex Rivera on 2026-03-14"*, then
*"Actually, cancel that"* quoting the reference - and check the database afterwards with:

```python
with sqlite3.connect(DB) as conn:
    print(conn.execute("SELECT * FROM bookings").fetchall())
```

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a HUGE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>